# Flywheel DICOM Download — Batch (All Participants)

This notebook loops through a list of participant IDs and downloads DICOM data for each one from UPenn Flywheel to the fMRI server.

**Before using this notebook**, run `flywheel_download_single.ipynb` at least once to:
- Create your `~/configs/config.ini` API key file
- Confirm your Flywheel group, project, and session labels are correct for your dataset

**Note:** You do not have to use this script. As long as your DICOMs are organized in the expected directory structure under `/fmriDataRaw/fmri_data_raw/{PROJECT}/{SUBJECT}/`, the rest of the pipeline will work regardless of how files were transferred.

---

### References
- [Flywheel Python SDK Docs](https://api-docs.flywheel.io/latest/index.html)
- [Flywheel Docs](https://docs.flywheel.io/)

## 1. Imports

In [ ]:
import flywheel
import tarfile
import zipfile
import os
import configparser

## 2. Connect to Flywheel

Reads your API key from `~/configs/config.ini`. If this file does not exist yet, run the setup cell in `flywheel_download_single.ipynb` first.

In [ ]:
# Read API key from config file
user = os.environ["USER"]
home_dir = f"/home/{user}"

config = configparser.ConfigParser()
config.read(f"{home_dir}/configs/config.ini")

if not config.has_option('UPENN-FLYWHEEL', 'apikey'):
    raise ValueError("API key not found. Run the setup cell in flywheel_download_single.ipynb first.")

api_key = config['UPENN-FLYWHEEL']['apikey']

# Initialize Flywheel client and confirm authentication
fw = flywheel.Client(api_key)

current_user = fw.get_current_user()
print(f"Connected to Flywheel as: {current_user.firstname} {current_user.lastname} ({current_user.email})")

## 3. Set Project & Participant Parameters

Edit the variables in this cell before running the loop.

**Prefix logic:** Subject IDs on Flywheel and on the server are constructed by combining a prefix with a number.  
For example, `in_prefix = 'sub'` and `sub = '001'` → Flywheel ID = `sub001`.  
The input and output prefixes can differ if your local naming convention differs from Flywheel's.

In [ ]:
# ── Flywheel identifiers ───────────────────────────────────────────────────────
group_label   = "your_group"        # Flywheel group label (e.g. lab name)
in_project    = "your_project"      # Flywheel project label
in_prefix     = "sub"               # Subject ID prefix as listed on Flywheel

# ── Local output identifiers ───────────────────────────────────────────────────
out_project   = "your_local_project"  # Project folder name on the fMRI server
out_prefix    = "sub"                 # Subject ID prefix for local storage

# ── Participant list ───────────────────────────────────────────────────────────
# List the numeric portion of each subject ID (without the prefix)
# e.g. for subjects sub001, sub002, sub003: subs = ['001', '002', '003']
subs = [
    '001',
    '002',
    '003',
    # add more subjects here...
]

# ── Derived paths ─────────────────────────────────────────────────────────────
outpath = f"/fmriDataRaw/fmri_data_raw/{out_project}"

print(f"Flywheel project : {group_label}/{in_project}")
print(f"Local output dir : {outpath}")
print(f"Subjects to download ({len(subs)}): {subs}")

## 4. Define Transfer Function

This function handles the full download-and-extract workflow for a single subject.  
If the subject's output directory already exists, the download is skipped — making it safe to re-run the loop after failures without re-downloading completed subjects.

In [ ]:
def transfer_data(sub, in_prefix, out_prefix, in_project, out_project, group_label):
    """
    Download and extract DICOM data for a single subject from Flywheel.

    Skips the subject if the output directory already exists, so re-running
    the loop after a partial failure will only process missing subjects.

    Parameters
    ----------
    sub          : str  — Subject number (without prefix), e.g. '001'
    in_prefix    : str  — Subject prefix on Flywheel, e.g. 'sub'
    out_prefix   : str  — Subject prefix for local storage, e.g. 'sub'
    in_project   : str  — Flywheel project label
    out_project  : str  — Local project folder name under /fmriDataRaw/fmri_data_raw/
    group_label  : str  — Flywheel group label
    """
    flywheel_id = in_prefix + sub
    local_id    = out_prefix + sub
    outpath     = f"/fmriDataRaw/fmri_data_raw/{out_project}"
    output_dicom_dir = os.path.join(outpath, local_id)
    tar_path    = f"./working_data/{local_id}.tar"

    # Skip if this subject has already been downloaded
    if os.path.exists(output_dicom_dir):
        print(f"--- Skipping {flywheel_id}: output directory already exists ---")
        return

    print(f"\n{'='*60}")
    print(f"Transferring: {flywheel_id}")
    print(f"{'='*60}")

    # ── Step 1: Look up session on Flywheel ───────────────────────────────────
    lookup_string = f"{group_label}/{in_project}/{flywheel_id}"
    print(f"Looking up: {lookup_string}")
    session = fw.lookup(lookup_string)

    # ── Step 2: Download session tarball to working_data/ ─────────────────────
    os.makedirs("working_data", exist_ok=True)
    print(f"Downloading tar to: {tar_path}")
    fw.download_tar(session, tar_path)

    # ── Step 3: Create output directory ───────────────────────────────────────
    os.makedirs(output_dicom_dir, exist_ok=True)
    print(f"Output directory: {output_dicom_dir}")

    # ── Step 4: Extract DICOM zips from the tarball ───────────────────────────
    extracted_count = 0
    with open(tar_path, 'rb') as f:
        tar_data = tarfile.open(fileobj=f, mode='r:')

        for member in tar_data.getmembers():
            if 'dicom.zip' in member.name:  # only extract DICOM zip files
                print(f"  Extracting: {member.name}")
                tfile = tar_data.extractfile(member.name)
                dicom_zip = zipfile.ZipFile(tfile, mode='r')
                dicom_zip.extractall(output_dicom_dir)
                extracted_count += 1

        tar_data.close()

    print(f"\nParticipant {sub} complete — {extracted_count} DICOM archive(s) extracted.")

## 5. Run the Batch Download

This cell loops through all subjects in `subs` and calls `transfer_data()` for each one.

- Subjects with an existing output directory are automatically **skipped**, so it is safe to re-run this cell after a failure
- If a single subject fails, the error will be printed and the loop will continue to the next subject
- Check the printed output after each run to confirm all subjects completed successfully

In [ ]:
failed_subs = []

for sub in subs:
    try:
        transfer_data(sub, in_prefix, out_prefix, in_project, out_project, group_label)
    except Exception as e:
        print(f"\n!! ERROR on subject {sub}: {e}")
        failed_subs.append(sub)

# Summary
print(f"\n{'='*60}")
print(f"Batch download complete.")
print(f"  Attempted : {len(subs)} subjects")
print(f"  Failed    : {len(failed_subs)} subjects")
if failed_subs:
    print(f"  Failed IDs: {failed_subs}")
    print("  Re-add these to the 'subs' list and re-run, or use flywheel_download_single.ipynb")

## 6. Verify Output

In [ ]:
# List all subject directories that were created under the output project folder
print(f"Contents of {outpath}:")
for item in sorted(os.listdir(outpath)):
    item_path = os.path.join(outpath, item)
    n_files = len(os.listdir(item_path)) if os.path.isdir(item_path) else 'n/a'
    print(f"  {item}  ({n_files} items)")

## 7. Clean Up (Optional)

The `.tar` files in `working_data/` can be safely deleted once you have verified all subjects extracted correctly. Flywheel retains the original data.

> ⚠️ **Double-check the path before running.** This will recursively delete everything in working_data/.

In [ ]:
# Uncomment and run to delete all tarballs after confirming successful extraction

# import shutil
# shutil.rmtree('./working_data/')
# print('working_data/ deleted.')